# 4th 미니프로젝트 코드 정리본 - Team 1(퍼스트)






In [2]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import fbeta_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE


In [3]:
train_path = 'train.parquet'
test_path = 'test.parquet'

for path, label in [(train_path, 'train'), (test_path, 'test')]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'찾을 수 없음: {label} -> {path}')

print('train file ->', train_path)
print('test file  ->', test_path)

df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

df.columns = df.columns.str.strip().str.lower()
test_df.columns = test_df.columns.str.strip().str.lower()

df['golden_ratio_br'] = df['borrowing dependency'] / (df['retained earnings to total assets'] + 1e-9)
test_df['golden_ratio_br'] = test_df['borrowing dependency'] / (test_df['retained earnings to total assets'] + 1e-9)

final_vars = [
    'persistent eps in the last four seasons', 'borrowing dependency', 'net income to total assets',
    'total income/total expense', 'net value growth rate', 'total debt/total net worth',
    'non-industry income and expenditure/revenue', 'net worth/assets', 'interest expense ratio',
    'retained earnings to total assets', 'equity to liability', 'after-tax net interest rate',
    'quick ratio', 'degree of financial leverage (dfl)', "net income to stockholder's equity",
    'interest coverage ratio (interest expense to ebit)', 'interest-bearing debt interest rate',
    'inventory/working capital', 'cash/current liability', 'golden_ratio_br'
]

X = df[final_vars]
y = df['bankrupt?']

print('train shape:', df.shape)
print('test shape :', test_df.shape)
print('positive rate:', y.mean())


train file -> train.parquet
test file  -> test.parquet
train shape: (4773, 98)
test shape : (2046, 97)
positive rate: 0.03226482296249738


## 1. 홀드아웃 평가용 학습/검증 분할

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

ratio = (y_train == 0).sum() / (y_train == 1).sum()

best_params = {
    'xgb_n_estimators': 712,
    'xgb_lr': 0.013410585240898247,
    'xgb_max_depth': 7,
    'xgb_subsample': 0.7774789438634512,
    'xgb_colsample': 0.7410703874761337,
    'lgb_n_estimators': 375,
    'lgb_lr': 0.01618709255825155,
    'lgb_max_depth': 8,
    'lgb_subsample': 0.9085133170965529,
    'lgb_colsample': 0.9359307444152808,
    'sampling_strategy': 0.11282883649924065
}

target_threshold = 0.03

print('class ratio (0/1):', ratio)
print('target threshold :', target_threshold)


class ratio (0/1): 30.040650406504064
target threshold : 0.03


In [5]:
def build_base_models(class_ratio, params):
    model_xgb = xgb.XGBClassifier(
        n_estimators=params['xgb_n_estimators'],
        learning_rate=params['xgb_lr'],
        max_depth=params['xgb_max_depth'],
        subsample=params['xgb_subsample'],
        colsample_bytree=params['xgb_colsample'],
        scale_pos_weight=class_ratio,
        random_state=42,
        verbosity=0
    )

    model_lgb = lgb.LGBMClassifier(
        n_estimators=params['lgb_n_estimators'],
        learning_rate=params['lgb_lr'],
        max_depth=params['lgb_max_depth'],
        subsample=params['lgb_subsample'],
        colsample_bytree=params['lgb_colsample'],
        class_weight='balanced',
        random_state=42,
        verbosity=-1
    )
    return model_xgb, model_lgb


def fit_stacking_model(X_train_part, y_train_part, X_eval_part, params, threshold):
    class_ratio = (y_train_part == 0).sum() / (y_train_part == 1).sum()

    smote = SMOTE(sampling_strategy=params['sampling_strategy'], random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train_part, y_train_part)

    model_xgb, model_lgb = build_base_models(class_ratio, params)
    model_xgb.fit(X_train_res, y_train_res)
    model_lgb.fit(X_train_res, y_train_res)

    meta_train_X = np.column_stack([
        model_xgb.predict_proba(X_train_res)[:, 1],
        model_lgb.predict_proba(X_train_res)[:, 1]
    ])

    meta_eval_X = np.column_stack([
        model_xgb.predict_proba(X_eval_part)[:, 1],
        model_lgb.predict_proba(X_eval_part)[:, 1]
    ])

    meta_model = LogisticRegression(class_weight='balanced', random_state=42)
    meta_model.fit(meta_train_X, y_train_res)

    eval_probs = meta_model.predict_proba(meta_eval_X)[:, 1]
    eval_preds = (eval_probs >= threshold).astype(int)

    return {
        'model_xgb': model_xgb,
        'model_lgb': model_lgb,
        'meta_model': meta_model,
        'X_train_res': X_train_res,
        'y_train_res': y_train_res,
        'eval_probs': eval_probs,
        'eval_preds': eval_preds
    }


In [6]:
holdout_run = fit_stacking_model(
    X_train_part=X_train,
    y_train_part=y_train,
    X_eval_part=X_test,
    params=best_params,
    threshold=target_threshold
)

current_f2 = fbeta_score(y_test, holdout_run['eval_preds'], beta=2, zero_division=0)
current_cm = confusion_matrix(y_test, holdout_run['eval_preds'])

print(f"F2 Score: {current_f2:.4f}")
print("\n[Confusion Matrix]")
print(current_cm)


F2 Score: 0.5526

[Confusion Matrix]
[[879  45]
 [ 10  21]]


## 2. 5-Fold 교차검증

원본 코드의 의도를 유지하되, 각 fold마다 모델을 새로 생성해서 CV 결과가 이후 셀의 test 예측에 영향을 주지 않도록 분리했습니다.


In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []
train_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

    fold_run = fit_stacking_model(
        X_train_part=X_train_fold,
        y_train_part=y_train_fold,
        X_eval_part=X_val_fold,
        params={
            **best_params,
            'sampling_strategy': 0.1128  # 원본 CV 셀과 동일
        },
        threshold=target_threshold
    )

    train_meta = np.column_stack([
        fold_run['model_xgb'].predict_proba(fold_run['X_train_res'])[:, 1],
        fold_run['model_lgb'].predict_proba(fold_run['X_train_res'])[:, 1]
    ])
    train_probs = fold_run['meta_model'].predict_proba(train_meta)[:, 1]
    train_preds = (train_probs >= target_threshold).astype(int)

    train_f2 = fbeta_score(fold_run['y_train_res'], train_preds, beta=2, zero_division=0)
    val_f2 = fbeta_score(y_val_fold, fold_run['eval_preds'], beta=2, zero_division=0)

    train_scores.append(train_f2)
    fold_scores.append(val_f2)

    print(f"Fold {fold}: Train F2 = {train_f2:.4f} | Val F2 = {val_f2:.4f}")

print("\n" + "=" * 30)
print(f"평균 Train F2: {np.mean(train_scores):.4f}")
print(f"평균 CV Score (Val F2): {np.mean(fold_scores):.4f} (±{np.std(fold_scores):.4f})")
print("=" * 30)


Fold 1: Train F2 = 0.9634 | Val F2 = 0.4396
Fold 2: Train F2 = 0.9616 | Val F2 = 0.5612
Fold 3: Train F2 = 0.9537 | Val F2 = 0.5263
Fold 4: Train F2 = 0.9630 | Val F2 = 0.4324
Fold 5: Train F2 = 0.9661 | Val F2 = 0.5978

평균 Train F2: 0.9616
평균 CV Score (Val F2): 0.5115 (±0.0657)


## 3. 제출용 test 예측

competition workflow 기준으로는 **CV용 모델이 아니라**, 홀드아웃 기준 최종 학습 모델을 사용해 test를 예측합니다.


In [8]:
X_real_test = test_df[final_vars]
test_ids = test_df['id']

final_model_run = fit_stacking_model(
    X_train_part=X_train,
    y_train_part=y_train,
    X_eval_part=X_real_test,
    params=best_params,
    threshold=target_threshold
)

real_preds = final_model_run['eval_preds']

result_df = pd.DataFrame({
    'ID': test_ids,
    'Bankrupt?': real_preds
})

result_df.to_csv('result_competition_workflow.csv', index=False)
print(f"예측된 부도 기업 수: {real_preds.sum()}개 / 전체: {len(real_preds)}개")
print('saved -> result_competition_workflow.csv')


예측된 부도 기업 수: 135개 / 전체: 2046개
saved -> result_competition_workflow.csv


## 4. 모델 해석용 랜덤포레스트 중요도

In [9]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)
rf.fit(X, y)

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

print("RF 중요도 Top 15:")
for col, val in imp.head(15).items():
    print(f"  {col[:45]:<45} {val:.4f}")


RF 중요도 Top 15:
  golden_ratio_br                               0.1417
  persistent eps in the last four seasons       0.0828
  net income to total assets                    0.0637
  after-tax net interest rate                   0.0593
  total income/total expense                    0.0575
  total debt/total net worth                    0.0568
  retained earnings to total assets             0.0565
  borrowing dependency                          0.0539
  net value growth rate                         0.0489
  net worth/assets                              0.0483
  quick ratio                                   0.0439
  non-industry income and expenditure/revenue   0.0433
  equity to liability                           0.0423
  interest expense ratio                        0.0412
  degree of financial leverage (dfl)            0.0347


## 5. 핵심 숫자 정리

In [10]:
summary_df = pd.DataFrame({
    '항목': [
        '홀드아웃 F2',
        '홀드아웃 정밀도',
        '홀드아웃 재현율',
        'CV 평균 Train F2',
        'CV 평균 Val F2',
        'CV Val 표준편차'
    ],
    '값': [
        round(current_f2, 4),
        round((current_cm[1,1] / (current_cm[1,1] + current_cm[0,1])) if (current_cm[1,1] + current_cm[0,1]) > 0 else 0, 4),
        round((current_cm[1,1] / (current_cm[1,1] + current_cm[1,0])) if (current_cm[1,1] + current_cm[1,0]) > 0 else 0, 4),
        round(float(np.mean(train_scores)), 4),
        round(float(np.mean(fold_scores)), 4),
        round(float(np.std(fold_scores)), 4)
    ]
})
summary_df


,항목,값
0,홀드아웃 F2,0.5526
1,홀드아웃 정밀도,0.3182
2,홀드아웃 재현율,0.6774
3,CV 평균 Train F2,0.9616
4,CV 평균 Val F2,0.5115
5,CV Val 표준편차,0.0657
